# General Cancer Detection System — CNN (Image Data)

A **general-purpose** CNN pipeline for cancer detection from images. Works with *any* binary cancer image dataset on Kaggle — skin lesions, histopathology slides, lung CT scans, brain MRIs, etc. — as long as images are organized into two folders (cancer / no-cancer).

**Example compatible Kaggle datasets:**
- Skin Cancer (HAM10000 / ISIC)
- Breast Cancer Histopathology (IDC)
- Lung & Colon Cancer Histopathological Images
- Brain Tumor MRI

>  **Disclaimer:** Educational project only — never use for real medical diagnosis.

**Expected folder structure:**
```
data/
├── cancer/        (or 'malignant', 'positive', etc.)
│   ├── img1.jpg
│   └── ...
└── no_cancer/      (or 'benign', 'negative', etc.)
    ├── img1.jpg
    └── ...
```

##  Step 1: Install & Import Libraries

In [ ]:
# !pip install tensorflow scikit-learn matplotlib seaborn kaggle opencv-python Pillow

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from PIL import Image
import cv2

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')
print('Libraries imported!')

##  Step 2: Download a Dataset from Kaggle

Pick **any one** dataset below by uncommenting it, or substitute your own.

**Prerequisites:** Kaggle account → Account → API → Create New Token → place `kaggle.json` in `~/.kaggle/`

In [ ]:
# Uncomment if kaggle.json isn't in the default location
# os.environ['KAGGLE_USERNAME'] = 'your_kaggle_username'
# os.environ['KAGGLE_KEY'] = 'your_kaggle_api_key'

# Choose ONE dataset (uncomment one line):

# Option A: Skin Cancer (HAM10000)
# !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 --unzip -p ./data

# Option B: Breast Histopathology Images (IDC)
# !kaggle datasets download -d paultimothymooney/breast-histopathology-images --unzip -p ./data

# Option C: Lung and Colon Cancer Histopathological Images
!kaggle datasets download -d andrewmvd/lung-and-colon-cancer-histopathological-images --unzip -p ./data

print('Dataset downloaded! Check ./data and update DATA_DIR / class folder names below.')

## Step 3: Configure Dataset Paths

> **Important:** Different Kaggle datasets use different folder structures. After downloading, inspect `./data` and update `DATA_DIR` below to point to the folder that directly contains your class subfolders.

In [ ]:
# Inspect what was downloaded
for root, dirs, files in os.walk('./data'):
    level = root.replace('./data', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for d in dirs[:10]:
            print(f'{indent}  {d}/')
    if level >= 2:
        break

In [ ]:
#  UPDATE THIS to point to the folder containing your two class subfolders
DATA_DIR = './data'

classes = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
print(f'Detected classes: {classes}')

if len(classes) != 2:
    print('Expected exactly 2 classes for binary cancer/no-cancer detection.')
    print('   Update DATA_DIR above to point to the correct folder level.')
else:
    counts = {cls: len(os.listdir(os.path.join(DATA_DIR, cls))) for cls in classes}
    for cls, count in counts.items():
        print(f'  {cls}: {count} images')

##  Step 4: Explore Sample Images

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
fig.suptitle('Sample Images per Class', fontsize=14, fontweight='bold')

for row, cls in enumerate(classes):
    cls_path = os.path.join(DATA_DIR, cls)
    sample_files = os.listdir(cls_path)[:6]
    for col, fname in enumerate(sample_files):
        img = Image.open(os.path.join(cls_path, fname)).convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].set_title(cls, fontsize=9)
        axes[row, col].axis('off')

plt.tight_layout()
plt.show()

##  Step 5: Preprocess & Augment Data

In [ ]:
IMG_SIZE   = (128, 128)
BATCH_SIZE = 32
SEED       = 42

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,        # Medical images often have no fixed orientation
    zoom_range=0.1,
    brightness_range=[0.85, 1.15],
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', subset='training', seed=SEED
)

val_generator = val_datagen.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', subset='validation', seed=SEED
)

print(f'Class mapping: {train_generator.class_indices}')
print(f'Training samples  : {train_generator.samples}')
print(f'Validation samples: {val_generator.samples}')

##  Step 6: Build the CNN

In [ ]:
def build_cancer_cnn(input_shape):
    model = keras.Sequential([

        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),

        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

INPUT_SHAPE = (*IMG_SIZE, 3)
model = build_cancer_cnn(INPUT_SHAPE)
model.summary()

##  Step 7: Compile the Model

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)

print(' Model compiled!')

##  Step 8: Train the Model

In [ ]:
# Handle class imbalance
n_class1 = np.sum(train_generator.classes == 1)
n_class0 = np.sum(train_generator.classes == 0)
total = n_class1 + n_class0

class_weight = {
    0: total / (2 * n_class0),
    1: total / (2 * n_class1)
}
print(f'Class weights: {class_weight}')

callbacks = [
    EarlyStopping(monitor='val_auc', patience=10, restore_best_weights=True, mode='max', verbose=1),
    ModelCheckpoint('best_cancer_cnn.keras', monitor='val_auc', save_best_only=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)
]

EPOCHS = 40

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

print('\n Training complete!')

##  Step 9: Visualize Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training History', fontsize=15, fontweight='bold')
epochs_ran = range(1, len(history.history['accuracy']) + 1)

metrics = [('accuracy', 'val_accuracy', 'Accuracy'),
           ('loss', 'val_loss', 'Loss'),
           ('auc', 'val_auc', 'AUC')]

for ax, (tm, vm, title) in zip(axes, metrics):
    ax.plot(epochs_ran, history.history[tm], label='Train', linewidth=2)
    ax.plot(epochs_ran, history.history[vm], label='Val', linewidth=2, linestyle='--')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best Val AUC: {max(history.history['val_auc']):.4f}")

##  Step 10: Evaluate the Model

In [ ]:
best_model = keras.models.load_model('best_cancer_cnn.keras')

val_generator.reset()
y_pred_prob = best_model.predict(val_generator, verbose=1).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)
y_true = val_generator.classes
class_names = list(val_generator.class_indices.keys())

print('\n📋 Classification Report:')
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Evaluation', fontsize=14, fontweight='bold')

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=ax1,
            xticklabels=class_names, yticklabels=class_names)
ax1.set_title('Confusion Matrix')
ax1.set_ylabel('Actual')
ax1.set_xlabel('Predicted')

fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
roc_auc = auc(fpr, tpr)
ax2.plot(fpr, tpr, color='#e74c3c', linewidth=2, label=f'ROC (AUC = {roc_auc:.3f})')
ax2.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random')
ax2.fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

##  Step 11: Save the Model

In [ ]:
best_model.save('cancer_detection_cnn_final.keras')
print('Model saved as cancer_detection_cnn_final.keras')